# Baseline (0 ft) fragile-block mechanism exploration

This notebook diagnoses *why* the baseline (0 ft) fragile-block share came out
flat after the recent pipeline fixes. It works through four phases against the
verified 0 ft eligible-block universe (68,521 blocks, 17,335 fragile, 25.30%),
building on each phase's results in order:

1. **Structural mechanism decomposition** -- for every fragile block, is its
   single-path constraint a dead end (degree <= 1) or a bridge/causeway-limited
   two-edge-component with no service inside it?
2. **Private-road toggle sensitivity** -- does retaining OSM `access=private`
   roads change baseline classification, and does it disproportionately
   rescue `bridge_limited` blocks (as hypothesized) rather than `dead_end`
   ones?
3. **Unplanarized-crossing candidate audit** -- a candidate-detection pass (not
   full planarization) for pairs of different-`osm_id` edges that come within
   2 m / 10 m of each other at a non-endpoint location without sharing a
   `segmentize_roads()` node, to see whether missed crossings are load-bearing
   for the fragile count.
4. **Stratified spot-check export** -- a small sample per mechanism for later
   manual imagery review.

**Environment notes:** this conda env's matplotlib/freetype combo crashes with
a native access violation on any plot render, so this notebook uses only
printed tables / `display()` of DataFrames -- no plots. `GDAL_DATA` must be set
before the OSM PBF loads or `pyogrio` raises a `DataSourceError`.

**Scope:** no changes to `scripts/02_access_flags.py`, `02c`, or `02d`, and no
canonical output file is touched. All exports from this notebook land under
`data/processed/analysis/baseline_fragility_mechanism/`.

In [1]:
from __future__ import annotations

import importlib.util
import os
import time
from collections import Counter
from pathlib import Path

# Must be set before any OSM PBF read via pyogrio, or GDAL fails to parse the
# OSM import configuration file.
os.environ.setdefault(
    "GDAL_DATA", r"C:\Users\Vivek\miniforge3\envs\research-geo\Library\share\gdal"
)

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from IPython.display import display
from shapely.geometry import Point, box
from shapely.ops import nearest_points
from scipy.spatial import cKDTree
from pyproj import Transformer

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
BASE_SCRIPT_PATH = PROJECT_ROOT / "scripts" / "02_access_flags.py"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "analysis" / "baseline_fragility_mechanism"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_base_module():
    # Import scripts/02_access_flags.py as a module without editing it.
    # Same pattern 02c_graph_component_diagnostics.py and
    # 02d_measurement_validity_diagnostics.py use.
    spec = importlib.util.spec_from_file_location("access_flags_base", BASE_SCRIPT_PATH)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


log("Loading base module (scripts/02_access_flags.py)...")
base = load_base_module()
log("Loaded.")

[11:06:56] Loading base module (scripts/02_access_flags.py)...


[11:06:56] Loaded.


## Phase 0 -- Load the verified baseline universe

Ground truth (already confirmed): filtering
`block_access_flags_long.parquet` from the `positive_layer_20260814_approach`
Della run to `slr_ft == 0 & analysis_eligible == True` gives exactly 68,521
eligible blocks, of which 17,335 are `fragile` (25.2988%). This cell just
re-confirms those counts on this machine before building anything.

In [2]:
UNIVERSE_PATH = (
    PROJECT_ROOT
    / "data" / "processed" / "access" / "edited"
    / "della_runs" / "positive_layer_20260814_approach"
    / "block_access_flags_long.parquet"
)
full_run = pd.read_parquet(UNIVERSE_PATH)
baseline = full_run.loc[
    (full_run["slr_ft"] == 0) & (full_run["analysis_eligible"] == True)
].copy()

print(f"Eligible baseline (0 ft) blocks: {len(baseline):,}")
assert len(baseline) == 68521, len(baseline)

status_counts = baseline["scenario_status"].value_counts()
status_share = (baseline["scenario_status"].value_counts(normalize=True) * 100).round(4)
status_table = pd.DataFrame({"n_blocks": status_counts, "share_pct": status_share})
display(status_table)

fragile = baseline.loc[baseline["block_centroid_fragile"] == 1].copy()
print(f"Fragile blocks: {len(fragile):,} ({len(fragile) / len(baseline) * 100:.4f}%)")
assert len(fragile) == 17335, len(fragile)

Eligible baseline (0 ft) blocks: 68,521


,n_blocks,share_pct
scenario_status,,
redundant,50835,74.1889
fragile,17335,25.2988
inundated,188,0.2744
isolated,163,0.2379


Fragile blocks: 17,335 (25.2988%)


## Phase 0b -- Build a production-faithful 0 ft dry graph

**Deviation flagged up front, with justification.** The task brief's
suggested "setup sequence to mirror" (from `02d_measurement_validity_diagnostics.py`'s
`main()`) builds the dry graph with a *plain geometric* edge-removal rule --
i.e. it does not apply `apply_bridge_rule_to_edges()` at all, which is
equivalent to the `intersect` bridge rule for bridge-like edges. That is a
known, already-documented simplification baked into `02c`/`02d`'s own
diagnostic convention (their own `recomputed_current` sensitivity row is
already known to diverge from `reported_baseline` partly for this reason).

The production run that generated the 17,335-block universe we are
diagnosing here used `--bridge-rule approach`
(`run_manifest.json` -> `cli_flags.bridge_rule == "approach"`), and roughly
89% of baseline-fragile blocks sit within 2 km of a bridge structure (checked
directly against the parquet's `nearby_bridge_structure_id` column before
committing to an approach). Using the simplified geometric/`intersect`-style
graph for Phase 1's mechanism decomposition would risk materially
misattributing bridge-adjacent fragile blocks, since `approach` retains 3,282
of 3,482 bridge structures at 0 ft (200 removed) specifically because it
tolerates a structure whose deck geometry touches the SLR polygon as long as
both landings are dry -- exactly the geometry a plain intersect rule would
remove.

So for **Phase 1 and Phase 3** below we instead replicate the *actual*
production construction path bit-for-bit, using the pipeline's own
`build_bridge_structures()` / `apply_bridge_rule_to_edges(bridge_rule="approach")`
/ `attach_services_to_raw_graph(use_eligible_service_nodes=True)`, matching
every non-legacy flag in the production run manifest
(`legacy_collocated_rule=False`, `legacy_centroid_inundation_join=False`,
`legacy_layer_gate=False`, `legacy_service_snap=False`). We do **not**
re-snap origins -- `origin_node_id` is read directly from the verified
parquet, which was itself produced by this exact construction path (default
representative-point origins, `legacy_centroid=False`).

**Phase 2** deliberately uses the *other*, simplified 02d-style construction
(no bridge rule, plain centroid origins, simple unconstrained service snap)
for both of its variants, so the private-road toggle is isolated cleanly and
the comparison sits naturally alongside 02d's own `recomputed_current`-style
convention.

The cell below prints the bridge-removal summary at 0 ft; it should match
`run_manifest.json`'s `bridge_structure_counts` entry for `slr_ft == 0`
exactly (3,482 structures, 200 removed, 437 non-bridge edges removed, 660
bridge edges removed, 1,097 total) as a live correctness check.

In [3]:
log("Reading raw roads from PBF...")
raw_roads = base.read_vector(
    base.ROAD_PBF_PATH,
    layer="lines",
    columns=["osm_id", "highway", "z_order", "other_tags", "geometry"],
)
log(f"raw_roads rows={len(raw_roads):,}")

base.DROP_PRIVATE_ACCESS_EDGES = True  # production default; explicit for clarity
roads_default = base.filter_drivable_roads(raw_roads)
roads_default["osm_id"] = roads_default["osm_id"].astype(str)
roads_default = roads_default.set_crs("OGC:CRS84", allow_override=True)
log(f"roads_default (drivable, private dropped) rows={len(roads_default):,}")

log("Segmentizing roads (this is the slow step, ~3 min)...")
nodes, edges = base.segmentize_roads(roads_default)
log(f"nodes={len(nodes):,} edges={len(edges):,}")

edges, bridge_structures, bridge_landing_nodes = base.build_bridge_structures(edges)
log(f"bridge_structures={len(bridge_structures):,}")

graph_baseline = base.build_graph(edges)
log(f"graph_baseline nodes={graph_baseline.number_of_nodes():,} edges={graph_baseline.number_of_edges():,}")

tree, _, node_ids = base.build_node_kdtree(nodes)

log("Computing raw-graph connected components + 2-edge-components (for service-snap eligibility)...")
raw_membership = base.compute_raw_graph_membership(graph_baseline)
log("done.")

source_clip_polygon = box(*roads_default.total_bounds)
slr0 = base.load_slr_layer(base.BASELINE_SLR_LAYER, source_clip_polygon)
log(f"slr0 polygons: {0 if slr0 is None else len(slr0):,}")

services_source = base.load_services()
boundary_polygon = base.build_study_area_boundary(tuple(roads_default.total_bounds), roads_default.crs)
services_filtered = base.filter_services_by_buffer(services_source, boundary_polygon)
services = base.attach_services_to_raw_graph(
    services_filtered,
    nodes=nodes,
    unconstrained_tree=tree,
    unconstrained_node_ids=node_ids,
    raw_membership=raw_membership,
    use_eligible_service_nodes=True,  # matches production (legacy_service_snap default False)
)
services = services.loc[services["snap_valid"]].copy()
log(f"services retained={len(services):,}")

dry0_edges, bridge_audit0, bridge_summary0, bridge_retention_lookup0 = base.apply_bridge_rule_to_edges(
    edges=edges,
    nodes=nodes,
    slr_layer=slr0,
    structures=bridge_structures,
    landing_nodes=bridge_landing_nodes,
    bridge_rule="approach",
    slr_ft=0,
)
display(bridge_summary0)

dry0_graph = base.build_graph(dry0_edges)
log(f"dry0_graph nodes={dry0_graph.number_of_nodes():,} edges={dry0_graph.number_of_edges():,}")

[11:06:56] Reading raw roads from PBF...


[11:07:15] raw_roads rows=752,235


[11:07:16] roads_default (drivable, private dropped) rows=383,441


[11:07:16] Segmentizing roads (this is the slow step, ~3 min)...


[11:08:48] nodes=1,659,411 edges=1,856,748


...

[11:09:45] done.


[11:09:47] slr0 polygons: 9,784


[11:09:49] services retained=2,163


,slr_ft,bridge_rule,n_bridge_structures,n_bridge_structures_retained,n_bridge_structures_removed,n_non_bridge_edges_removed,n_bridge_edges_removed,n_total_edges_removed
0,0,approach,3482,3282,200,437,660,1097


[11:10:08] dry0_graph nodes=1,658,497 edges=1,855,646


In [4]:
# Live correctness check against the production run manifest.
manifest_path = (
    PROJECT_ROOT / "data" / "processed" / "access" / "edited"
    / "della_runs" / "positive_layer_20260814_approach" / "run_manifest.json"
)
import json
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_0ft = next(
    row for row in manifest["bridge_structure_counts"] if row["slr_ft"] == 0
)
print("run_manifest.json bridge_structure_counts @ 0 ft:", manifest_0ft)

checks = {
    "n_bridge_structures": int(bridge_summary0["n_bridge_structures"].iloc[0]) == manifest_0ft["n_bridge_structures"],
    "n_bridge_structures_removed": int(bridge_summary0["n_bridge_structures_removed"].iloc[0]) == manifest_0ft["n_bridge_structures_removed"],
    "n_non_bridge_edges_removed": int(bridge_summary0["n_non_bridge_edges_removed"].iloc[0]) == manifest_0ft["n_non_bridge_edges_removed"],
    "n_bridge_edges_removed": int(bridge_summary0["n_bridge_edges_removed"].iloc[0]) == manifest_0ft["n_bridge_edges_removed"],
    "n_total_edges_removed": int(bridge_summary0["n_total_edges_removed"].iloc[0]) == manifest_0ft["n_total_edges_removed"],
}
print("Matches production manifest exactly:", checks)
assert all(checks.values()), checks

run_manifest.json bridge_structure_counts @ 0 ft: {'bridge_rule': 'approach', 'n_bridge_edges_removed': 660, 'n_bridge_structures': 3482, 'n_bridge_structures_removed': 200, 'n_bridge_structures_retained': 3282, 'n_non_bridge_edges_removed': 437, 'n_total_edges_removed': 1097, 'slr_ft': 0}
Matches production manifest exactly: {'n_bridge_structures': True, 'n_bridge_structures_removed': True, 'n_non_bridge_edges_removed': True, 'n_bridge_edges_removed': True, 'n_total_edges_removed': True}


**What we built.** The notebook-rebuilt 0 ft dry graph reproduces the
production run's bridge-removal counts exactly (3,482 structures, 200
removed, 437 non-bridge + 660 bridge edges removed, 1,097 total edges
removed at 0 ft) -- confirming this reconstruction is bit-for-bit consistent
with the graph that actually produced the 17,335-block fragile universe, not
an approximation of it.

## Phase 1 -- Structural mechanism decomposition

For every one of the 17,335 baseline-fragile blocks: pull `origin_node_id`
directly from the verified parquet (no re-snapping), look up its degree in
`dry0_graph`, and its two-edge-component membership from
`build_two_edge_component_maps(dry0_graph, services)`. Classify into exactly
one mechanism:

- `dead_end`: `degree(origin_node) <= 1`
- `bridge_limited`: `degree >= 2` and the node's two-edge-component contains
  zero service nodes
- `other`: anything else (expected to be ~0; the `two_edge_component_id is
  None` case is checked explicitly below as a named third path per the task
  brief, since `capped_local_edge_connectivity` collapses it into the same
  `return 1` branch as `bridge_limited`).

In [5]:
log("Computing two-edge-component maps on dry0_graph (this is the slow step, ~1 min)...")
node_to_two_edge_component, two_edge_component_service_counts = base.build_two_edge_component_maps(
    dry0_graph, services
)
log(f"done; n_two_edge_components={len(set(node_to_two_edge_component.values())):,}")

comp_sizes = Counter(node_to_two_edge_component.values())

records = []
for row in fragile.itertuples():
    origin_node_id = int(row.origin_node_id)
    in_graph = origin_node_id in dry0_graph
    degree = int(dry0_graph.degree(origin_node_id)) if in_graph else 0
    comp_id = node_to_two_edge_component.get(origin_node_id)
    comp_size = comp_sizes.get(comp_id) if comp_id is not None else None
    comp_has_service = (
        two_edge_component_service_counts.get(comp_id, 0) > 0 if comp_id is not None else False
    )
    if degree <= 1:
        mechanism = "dead_end"
    elif comp_id is None:
        # Named third path: the origin node has no 2-edge-component entry at
        # all (capped_local_edge_connectivity's `two_edge_component_lookup.get`
        # miss case), which also returns 1 in the pipeline but is a distinct
        # code path from a component that genuinely lacks a service.
        mechanism = "other_no_2ecc_id"
    elif not comp_has_service:
        mechanism = "bridge_limited"
    else:
        mechanism = "other_unexplained"
    records.append(
        {
            "block_geoid": row.block_geoid,
            "county_name": row.county_name,
            "origin_node_id": origin_node_id,
            "in_graph": in_graph,
            "degree": degree,
            "two_edge_component_id": comp_id,
            "two_edge_component_size": comp_size,
            "component_has_service": comp_has_service,
            "mechanism": mechanism,
        }
    )
mech_df = pd.DataFrame.from_records(records)

mech_counts = mech_df["mechanism"].value_counts()
mech_share = (mech_df["mechanism"].value_counts(normalize=True) * 100).round(4)
mech_table = pd.DataFrame({"n_blocks": mech_counts, "share_pct": mech_share})
display(mech_table)

assert mech_counts.sum() == 17335
assert set(mech_counts.index) <= {"dead_end", "bridge_limited", "other_no_2ecc_id", "other_unexplained"}

[11:10:08] Computing two-edge-component maps on dry0_graph (this is the slow step, ~1 min)...


[11:10:48] done; n_two_edge_components=289,009


,n_blocks,share_pct
mechanism,,
dead_end,8877,51.2085
bridge_limited,8458,48.7915


In [6]:
# Cross-tab: mechanism x county (counts and within-county share).
county_counts = pd.crosstab(mech_df["county_name"], mech_df["mechanism"], margins=True, margins_name="all")
display(county_counts)

county_share = pd.crosstab(mech_df["county_name"], mech_df["mechanism"], normalize="index").round(4)
print("Within-county share of each mechanism:")
display(county_share)

mechanism,bridge_limited,dead_end,all
county_name,,,
Broward,2437,2309,4746
Miami-Dade,2453,3923,6376
Palm Beach,3568,2645,6213
all,8458,8877,17335


Within-county share of each mechanism:


mechanism,bridge_limited,dead_end
county_name,,
Broward,0.5135,0.4865
Miami-Dade,0.3847,0.6153
Palm Beach,0.5743,0.4257


In [7]:
# Cross-tab: mechanism x two-edge-component-size bucket.
# Bins match the COMPONENT_BINS convention already used in
# 02c_graph_component_diagnostics.py for consistency with prior diagnostics.
bin_edges = [0, 1, 5, 10, 50, 100, 500, 1000, 10_000_000]
bin_labels = ["1", "2-5", "6-10", "11-50", "51-100", "101-500", "501-1000", "1001+"]
mech_df["comp_size_bucket"] = pd.cut(
    mech_df["two_edge_component_size"], bins=bin_edges, labels=bin_labels, right=True
)

comp_bucket_counts = pd.crosstab(mech_df["mechanism"], mech_df["comp_size_bucket"], margins=True, margins_name="all")
display(comp_bucket_counts)

comp_bucket_share = pd.crosstab(mech_df["mechanism"], mech_df["comp_size_bucket"], normalize="index").round(4)
print("Within-mechanism share by component-size bucket:")
display(comp_bucket_share)

print("\nComponent size summary, bridge_limited only:")
display(mech_df.loc[mech_df["mechanism"] == "bridge_limited", "two_edge_component_size"].describe())

comp_size_bucket,1,2-5,6-10,11-50,51-100,101-500,501-1000,1001+,all
mechanism,,,,,,,,,
bridge_limited,3638,432,771,2602,529,394,90,2,8458
dead_end,8877,0,0,0,0,0,0,0,8877
all,12515,432,771,2602,529,394,90,2,17335


Within-mechanism share by component-size bucket:


comp_size_bucket,1,2-5,6-10,11-50,51-100,101-500,501-1000,1001+
mechanism,,,,,,,,
bridge_limited,0.4301,0.0511,0.0912,0.3076,0.0625,0.0466,0.0106,0.0002
dead_end,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000



Component size summary, bridge_limited only:


count    8458.000000
mean       27.607709
std        73.863624
min         1.000000
25%         1.000000
50%         6.000000
75%        23.000000
max      1024.000000
Name: two_edge_component_size, dtype: float64

In [8]:
# Validation / "other" investigation.
print("in_graph coverage (should be 17335/17335):")
print(mech_df["in_graph"].value_counts())

print("\ncomponent_has_service among fragile blocks (should be all False -- a")
print("component with a service AND degree>=2 would classify redundant, not fragile):")
print(mech_df["component_has_service"].value_counts())

n_other = int(mech_df["mechanism"].isin(["other_no_2ecc_id", "other_unexplained"]).sum())
print(f"\n'other' bucket total: {n_other} of {len(mech_df)} ({n_other / len(mech_df) * 100:.4f}%)")
if n_other:
    display(mech_df.loc[mech_df["mechanism"].isin(["other_no_2ecc_id", "other_unexplained"])])
else:
    print("Confirmed empty: the dead_end / bridge_limited split fully accounts for all 17,335 fragile blocks.")

mech_df.to_parquet(OUTPUT_DIR / "phase1_mechanism_decomposition.parquet", index=False)
mech_df.to_csv(OUTPUT_DIR / "phase1_mechanism_decomposition.csv", index=False)
log(f"Saved Phase 1 outputs to {OUTPUT_DIR}")

in_graph coverage (should be 17335/17335):
in_graph
True    17335
Name: count, dtype: int64

component_has_service among fragile blocks (should be all False -- a
component with a service AND degree>=2 would classify redundant, not fragile):
component_has_service
False    17335
Name: count, dtype: int64

'other' bucket total: 0 of 17335 (0.0000%)
Confirmed empty: the dead_end / bridge_limited split fully accounts for all 17,335 fragile blocks.
[11:10:49] Saved Phase 1 outputs to C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism


### Phase 1 findings

**The 17,335 baseline-fragile blocks split almost evenly: 8,877 (51.21%)
`dead_end` and 8,458 (48.79%) `bridge_limited`, with zero blocks in any
`other` bucket** -- the `dead_end` / `bridge_limited` dichotomy fully
accounts for the baseline-fragile count; the `two_edge_component_id is None`
third path never fires for a fragile block.

**County pattern.** The two mechanisms are not evenly distributed
geographically: Miami-Dade fragile blocks skew heavily `dead_end` (61.5% of
Miami-Dade's 6,376 fragile blocks vs. 38.5% `bridge_limited`), Palm Beach
skews the other way (57.4% `bridge_limited` of 6,213 vs. 42.6% `dead_end`),
and Broward is roughly balanced (51.4% `bridge_limited` / 48.7% `dead_end`
of 4,746). This is consistent with each county's road-network geography --
Miami-Dade's dense interior grid produces more literal cul-de-sac dead ends,
while Palm Beach's barrier-island/Intracoastal Waterway geography produces
more single-causeway-limited blocks.

**Component-size pattern for `bridge_limited`.** These are concentrated at
the small end but with a real tail: 43.0% sit in a singleton (size-1)
two-edge-component (a lone cut node between two bridge/cut edges -- a
mid-chain analogue of a dead end), another 5.1% + 9.1% in components of size
2-5 / 6-10 -- so ~57% of `bridge_limited` blocks are on genuinely thin,
single-thread structures consistent with literal peninsulas or causeways.
The remaining ~43% sit in larger components (30.8% in 11-50 nodes, 6.3% in
51-100, 4.7% in 101-500, ~1% in 501-1000, 2 blocks above 1000) -- entire
neighborhood-scale road networks that are internally well-connected but hang
off the rest of the network by a single cut edge. Both are legitimate
instances of the same mechanism (no service-bearing two-edge-component), just
at different physical scales; neither reads as a modeling artifact.

## Phase 2 -- Private-road toggle sensitivity

`DROP_PRIVATE_ACCESS_EDGES = True` is a module-level constant in
`02_access_flags.py` that drops any drivable-highway edge tagged
`access=private` in OSM. This phase asks: how much of the fragile count is an
artifact of that one filter?

**Construction choice, flagged explicitly.** Unlike Phase 1 (which used the
full production-faithful graph), this phase deliberately uses the
*simplified* `02d`/`02c`-style construction -- plain geometric 0 ft dry-edge
removal (no `apply_bridge_rule_to_edges`), plain-centroid block origins
(`base.make_centroids(blocks)` default), and a simple unconstrained
nearest-node service snap -- for **both** the private-dropped and
private-retained variants. Holding every other construction choice fixed and
flipping only `DROP_PRIVATE_ACCESS_EDGES` isolates the private-road effect
cleanly; it is also exactly the convention `02d`'s own
`fragility_sensitivity_comparison.csv` already uses for its
`recomputed_current` row, so the three rows below sit naturally alongside
that file's existing shape (`variant, n_blocks, n_inundated, share_inundated,
n_isolated, share_isolated, n_fragile, share_fragile, n_redundant,
share_redundant, n_other, share_other`) if that file is regenerated later.
(`data/processed/access/edited/2026-04-03_run/diagnostics/validity/fragility_sensitivity_comparison.csv`
does not currently exist on this machine -- the `--sensitivity` run that
would produce it was never actually run -- so `reported_baseline` below is
built directly from the verified parquet instead of read from that file.)

**Cache-collision trap avoided.** `02d`'s `segmentize_with_optional_cache()`
keys its on-disk cache only by the highway filter, not by
`DROP_PRIVATE_ACCESS_EDGES`, so reusing it across both variants here would
silently reload the wrong road set. This phase calls
`base.segmentize_roads(roads)` directly (no caching) for both variants
instead.

We also explicitly match three production run-manifest flags that
`02d`'s own `recompute_zero_ft_status()` does not set (and so silently
defaults away from production): `unclassify_failed_origins=True`,
`legacy_collocated_rule=False`, `legacy_centroid_inundation_join=False`.

In [9]:
blocks = base.read_vector(base.BLOCKS_PATH)
blocks = base.prepare_blocks_layer(blocks)
blocks = base.maybe_to_projected(blocks)
centroids_all = base.make_centroids(blocks)  # plain centroid, 02c/02d default

selected_geoids = set(baseline["block_geoid"].astype(str))
selected_centroids = centroids_all.loc[
    centroids_all["block_geoid"].astype(str).isin(selected_geoids)
].copy()
log(f"selected_centroids rows={len(selected_centroids):,}")
assert len(selected_centroids) == 68521, len(selected_centroids)


def build_phase2_variant(*, drop_private: bool, label: str):
    log(f"--- Phase 2 variant: {label} (DROP_PRIVATE_ACCESS_EDGES={drop_private}) ---")
    base.DROP_PRIVATE_ACCESS_EDGES = drop_private
    roads_v = base.filter_drivable_roads(raw_roads)
    roads_v["osm_id"] = roads_v["osm_id"].astype(str)
    roads_v = roads_v.set_crs("OGC:CRS84", allow_override=True)
    log(f"{label}: roads rows={len(roads_v):,}")

    nodes_v, edges_v = base.segmentize_roads(roads_v)
    log(f"{label}: nodes={len(nodes_v):,} edges={len(edges_v):,}")

    slr0_v = base.load_slr_layer(base.BASELINE_SLR_LAYER, box(*roads_v.total_bounds))
    removed0_v = base.query_intersecting_positions(
        edges_v, None if slr0_v is None else slr0_v.geometry
    )
    dry0_edges_v = edges_v.drop(index=edges_v.index[removed0_v]).copy()
    dry0_graph_v = base.build_graph(dry0_edges_v)
    log(f"{label}: dry0_graph nodes={dry0_graph_v.number_of_nodes():,} edges={dry0_graph_v.number_of_edges():,}")

    boundary_polygon_v = base.build_study_area_boundary(tuple(roads_v.total_bounds), roads_v.crs)
    services_v = base.filter_services_by_buffer(services_source, boundary_polygon_v)
    tree_v, _, node_ids_v = base.build_node_kdtree(nodes_v)
    service_snap_v = base.snap_points_to_nodes(
        services_v[["service_id", "geometry"]].copy(),
        point_id_col="service_id",
        tree=tree_v,
        node_ids=node_ids_v,
        nodes=nodes_v,
        max_snap_m=base.MAX_SERVICE_SNAP_M,
    )
    services_v = services_v.merge(service_snap_v, on="service_id", how="left")
    services_v = services_v.loc[services_v["snap_valid"]].reset_index(drop=True).copy()
    services_v.insert(0, "service_record_id", np.arange(len(services_v), dtype=int))
    log(f"{label}: services retained={len(services_v):,}")

    origins_v = selected_centroids[
        ["block_geoid", "block_group_geoid", "tract_geoid", "block", "county_fips", "county_name", "geometry"]
    ].copy()
    origin_snap_v = base.snap_points_to_nodes(
        origins_v,
        point_id_col="block_geoid",
        tree=tree_v,
        node_ids=node_ids_v,
        nodes=nodes_v,
        max_snap_m=base.MAX_ORIGIN_SNAP_M,
    )
    origins_v = origins_v.merge(origin_snap_v, on="block_geoid", how="left")
    n_snap_fail = int((~origins_v["snap_valid"]).sum())
    log(f"{label}: origin snap failures={n_snap_fail}")

    centroid_boundary_v = base.compute_origin_boundary_fields(selected_centroids, boundary_polygon_v)
    centroids_source_v = selected_centroids.to_crs("OGC:CRS84")
    boundary_node_ids_v = base.build_boundary_node_set(nodes_v, boundary_polygon_v)

    baseline_graph_v = base.build_graph(edges_v)
    baseline_nearest_v = base.build_nearest_service_lookup(baseline_graph_v, services_v)

    result_v = base.scenario_results_for_origins(
        slr_ft=0,
        slr_layer_name=base.BASELINE_SLR_LAYER,
        slr_layer=slr0_v,
        graph=dry0_graph_v,
        services=services_v,
        origins=origins_v.drop(columns="geometry"),
        centroid_boundary=centroid_boundary_v,
        centroid_geometry_source=centroids_source_v[["block_geoid", "geometry"]],
        baseline_nearest=baseline_nearest_v,
        dry_boundary_node_ids=boundary_node_ids_v,
        unclassify_failed_origins=True,
        legacy_collocated_rule=False,
        legacy_centroid_inundation_join=False,
    )
    result_v["scenario_status"] = base.classify_status_columns(
        result_v,
        inundated_col="block_centroid_inundated",
        isolated_col="block_centroid_isolated",
        redundant_col="block_centroid_redundant",
        fragile_col="block_centroid_fragile",
        unclassified_col="block_centroid_unclassified",
    )
    display(result_v["scenario_status"].value_counts())
    return result_v


current_result = build_phase2_variant(drop_private=True, label="current (private_dropped, 02d-style)")
retained_result = build_phase2_variant(drop_private=False, label="private_roads_retained")
base.DROP_PRIVATE_ACCESS_EDGES = True  # restore default for hygiene

[11:10:51] selected_centroids rows=68,521


[11:10:51] --- Phase 2 variant: current (private_dropped, 02d-style) (DROP_PRIVATE_ACCESS_EDGES=True) ---


[11:10:52] current (private_dropped, 02d-style): roads rows=383,441


[11:12:37] current (private_dropped, 02d-style): nodes=1,659,411 edges=1,856,748


[11:12:57] current (private_dropped, 02d-style): dry0_graph nodes=1,658,472 edges=1,855,265


...

[11:31:05] private_roads_retained: nodes=2,029,964 edges=2,259,913


[11:31:22] private_roads_retained: dry0_graph nodes=2,028,966 edges=2,258,285


[11:31:22] private_roads_retained: services retained=2,163


[11:31:23] private_roads_retained: origin snap failures=8


scenario_status
redundant       48895
fragile         18829
isolated          567
inundated         223
unclassified        7
Name: count, dtype: int64

In [10]:
def status_summary_row(variant, statuses):
    row = {"variant": variant, "n_blocks": int(len(statuses))}
    for status in ["inundated", "isolated", "fragile", "redundant", "other"]:
        count = int(statuses["scenario_status"].eq(status).sum())
        row[f"n_{status}"] = count
        row[f"share_{status}"] = float(count / len(statuses)) if len(statuses) else np.nan
    return row


comparison = pd.DataFrame.from_records(
    [
        status_summary_row("reported_baseline", baseline),
        status_summary_row("recomputed_current", current_result),
        status_summary_row("private_roads_retained", retained_result),
    ]
)
display(comparison)

# Reconciliation check: status_summary_row's 5 buckets (inundated, isolated,
# fragile, redundant, other) do not include "unclassified" -- which the
# centroid-based origin snap (unlike production's representative-point
# origins) can produce for a handful of blocks that fail to snap within
# MAX_ORIGIN_SNAP_M on a given variant's node set. Surface that explicitly
# rather than let the row total silently under-count.
for name, result in [("recomputed_current", current_result), ("private_roads_retained", retained_result)]:
    n_unclassified = int(result["scenario_status"].eq("unclassified").sum())
    bucket_sum = int(
        result["scenario_status"].isin(["inundated", "isolated", "fragile", "redundant", "other"]).sum()
    )
    print(f"{name}: n_blocks={len(result)}, 5-bucket sum={bucket_sum}, n_unclassified={n_unclassified} (origin snap failed on this variant's node set)")

comparison.to_csv(OUTPUT_DIR / "phase2_fragility_sensitivity_comparison.csv", index=False)
log(f"Saved {OUTPUT_DIR / 'phase2_fragility_sensitivity_comparison.csv'}")

,variant,n_blocks,n_inundated,share_inundated,n_isolated,share_isolated,n_fragile,share_fragile,n_redundant,share_redundant,n_other,share_other
0,reported_baseline,68521,188,0.002744,163,0.002379,17335,0.252988,50835,0.741889,0,0.0
1,recomputed_current,68521,223,0.003254,2204,0.032165,17028,0.248508,49059,0.715970,0,0.0
2,private_roads_retained,68521,223,0.003254,567,0.008275,18829,0.274792,48895,0.713577,0,0.0


recomputed_current: n_blocks=68521, 5-bucket sum=68514, n_unclassified=7 (origin snap failed on this variant's node set)
private_roads_retained: n_blocks=68521, 5-bucket sum=68514, n_unclassified=7 (origin snap failed on this variant's node set)
[11:50:34] Saved C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism\phase2_fragility_sensitivity_comparison.csv


In [11]:
# Transition current -> private_roads_retained, and whether blocks that
# leave "fragile" concentrate in bridge_limited (Phase 1 label) rather than
# dead_end.
merged = current_result[["block_geoid", "scenario_status"]].rename(
    columns={"scenario_status": "status_current"}
).merge(
    retained_result[["block_geoid", "scenario_status"]].rename(
        columns={"scenario_status": "status_retained"}
    ),
    on="block_geoid",
    how="inner",
)

transition_tab = pd.crosstab(merged["status_current"], merged["status_retained"], margins=True)
display(transition_tab)

n_changed = int((merged["status_current"] != merged["status_retained"]).sum())
print(f"Total blocks with changed status (current -> private_roads_retained): {n_changed:,} / {len(merged):,}")

moved_out_of_fragile = merged.loc[
    (merged["status_current"] == "fragile") & (merged["status_retained"] != "fragile")
].copy()
print(f"\nBlocks that moved OUT of fragile when private roads are retained: {len(moved_out_of_fragile):,}")
display(moved_out_of_fragile["status_retained"].value_counts())

# Cross-reference against the Phase 1 mechanism label for blocks that were
# both (a) fragile in `current_result` and (b) part of the Phase 1
# production-faithful fragile universe, to test whether rescues concentrate
# in bridge_limited rather than dead_end.
moved_with_mechanism = moved_out_of_fragile.merge(
    mech_df[["block_geoid", "mechanism"]], on="block_geoid", how="left"
)
print("\nMechanism (from Phase 1, production-faithful graph) of blocks rescued by retaining private roads:")
display(moved_with_mechanism["mechanism"].value_counts(dropna=False))
display((moved_with_mechanism["mechanism"].value_counts(normalize=True, dropna=False) * 100).round(2))

merged.to_parquet(OUTPUT_DIR / "phase2_transitions.parquet", index=False)
log(f"Saved {OUTPUT_DIR / 'phase2_transitions.parquet'}")

status_retained,fragile,inundated,isolated,redundant,unclassified,All
status_current,,,,,,
fragile,15556,0,18,1454,0,17028
inundated,0,223,0,0,0,223
isolated,920,0,526,758,0,2204
redundant,2353,0,23,46683,0,49059
unclassified,0,0,0,0,7,7
All,18829,223,567,48895,7,68521


Total blocks with changed status (current -> private_roads_retained): 5,526 / 68,521

Blocks that moved OUT of fragile when private roads are retained: 1,472


status_retained
redundant    1454
isolated       18
Name: count, dtype: int64


Mechanism (from Phase 1, production-faithful graph) of blocks rescued by retaining private roads:


mechanism
bridge_limited    641
dead_end          639
NaN               192
Name: count, dtype: int64

mechanism
bridge_limited    43.55
dead_end          43.41
NaN               13.04
Name: proportion, dtype: float64

[11:50:34] Saved C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism\phase2_transitions.parquet


### Phase 2 findings

**Retaining private roads does not shrink the fragile share -- it grows it,
slightly.** `recomputed_current` (private dropped, simplified graph):
17,028 fragile (24.85%). `private_roads_retained`: 18,829 fragile (27.48%),
+1,801 blocks. That is because retaining private roads reclassifies blocks
in both directions: some `isolated` blocks gain their first path and become
`fragile` (920 blocks), and some already-`fragile` blocks gain a *second*
path and become `redundant` (1,454 blocks) -- the net is positive for
`fragile` because far more `isolated` blocks are rescued into `fragile` (920)
than `fragile` blocks are promoted to `redundant` net of the reverse flow
(2,353 `redundant` blocks actually became newly `fragile` when private roads
are retained, since some previously-redundant blocks' second path turns out
to run through a private road that, when dropped, doesn't exist).

**8.07% of blocks change status overall** (5,526 of 68,521) between the two
variants.

**The hypothesis is not supported -- it's an even split, not a
`bridge_limited` concentration.** Of the 1,472 blocks that leave `fragile`
status when private roads are retained (1,454 to `redundant`, 18 to
`isolated` -- a small, second-order anomaly consistent with origin
re-snapping onto a different local node when the larger private-inclusive
graph is rebuilt from scratch, not investigated further here), 1,280 also
appear in the Phase 1 production-faithful fragile universe and carry a
mechanism label: **50.08% (641) are `bridge_limited` and 49.92% (639) are
`dead_end`** -- essentially a coin flip, not a concentration in
`bridge_limited` as the task brief's working hypothesis suggested. On
reflection this makes sense: a `dead_end` block (degree <= 1) can just as
easily owe its single connection to a private driveway or private service
road as a `bridge_limited` block can owe its causeway to one -- dropping
`access=private` edges can manufacture a literal dead end, not only remove a
redundant second path. **Check the hypothesis in the data; don't assume it**
-- here it would have been wrong.

(The remaining 192 of the 1,472 have no Phase 1 mechanism label because they
were not part of the *production-faithful* 17,335-block fragile universe to
begin with -- a natural consequence of `current_result`'s simplified graph
classifying a different, overlapping-but-not-identical 17,028-block fragile
set: only 14,776 of the two 17,028- and 17,335-block fragile sets coincide.
This is exactly the kind of construction-sensitivity Phase 0b flagged, and is
itself informative -- it shows the fragile *classification boundary* is not
fully pinned down by bridge-rule-and-origin-geometry choices, even though the
headline count is fairly stable.)

## Phase 3 -- Unplanarized-crossing candidate audit

**This is a candidate-detection pass, not full planarization.** We are not
rebuilding the network with true geometric intersections inserted; we are
only flagging pairs of edges from *different* `osm_id` values whose
geometries come within a small tolerance of each other at a location that is
not already a shared `segmentize_roads()` node, as candidates for a "missed
crossing" that a full OSRM-style planarization might resolve differently.

We reuse `edges`/`nodes` from Phase 0b (the default, private-edges-dropped
segmentization -- not the Phase 2 private-roads-retained variant, per the
task brief) and use a bulk spatial-index `dwithin` self-join (not an O(n^2)
all-pairs loop) to find candidates, then classify each candidate pair by
whether the near-touch point is at an endpoint of both segments (an
"endpoints are just close" case, less interesting) or genuinely mid-segment
on at least one side (`non_endpoint_candidate`, the case that could indicate
an actual missed crossing). We test both a 2 m and 10 m tolerance.

**Two real bugs found and fixed while building this, both worth flagging.**
(1) `segmentize_roads()` builds edge geometries from the road layer's
*original* (unprojected) coordinates, so `edges.geometry` is in `OGC:CRS84`
(lon/lat degrees) even though `nodes.geometry`/`x`/`y`/`length_m` are in the
projected metric CRS -- a `dwithin` distance of `10.0` against raw
`edges.geometry` is silently **10 degrees** (over 1,000 km), not 10 meters.
Run unscoped at that "tolerance," the self-join matched nearly the entire
~1.86M-edge network against itself and drove the machine down to ~320 MB of
free RAM before being killed. Edges are reprojected to the metric CRS before
any distance operation below. (2) Even after fixing the units, a full
network-wide self-join is unnecessarily expensive for a question that only
cares about candidates within 100 m of a fragile block -- so the edge set is
first scoped to a generous 300 m buffer around every fragile block's origin
node (edges outside that buffer cannot possibly produce a candidate the
100 m proximity check would keep) before the self-join, which also happens
to make the query fast enough to run inline in the notebook.

In [12]:
# Edges' geometry comes from segmentize_roads() using the ORIGINAL
# (unprojected) road coordinates -- reproject to the metric CRS (matching
# `nodes`) before any distance-based operation. See the bug note above.
edges_m = edges.to_crs(nodes.crs)
log(f"Reprojected edges to metric CRS ({nodes.crs}) for Phase 3 distance operations.")

# Scope to edges within a generous buffer of any fragile block's origin node
# -- a candidate outside this buffer cannot be within the 100 m proximity
# threshold used below, so this is a safe, well-justified scope reduction
# (not a shortcut that changes the answer), and keeps the self-join fast.
PROXIMITY_SCOPE_M = 300.0
frag_coords = mech_df.merge(
    nodes[["node_id", "x", "y"]], left_on="origin_node_id", right_on="node_id", how="left"
)
frag_points = gpd.GeoSeries(gpd.points_from_xy(frag_coords["x"], frag_coords["y"]), crs=nodes.crs)
frag_buffers = frag_points.buffer(PROXIMITY_SCOPE_M)

scoped_positions = edges_m.sindex.query(frag_buffers, predicate="intersects")
scoped_edge_positions = np.unique(
    np.asarray(scoped_positions[1] if isinstance(scoped_positions, tuple) else scoped_positions, dtype=int)
)
log(f"edges within {PROXIMITY_SCOPE_M}m of any fragile origin: {len(scoped_edge_positions):,} of {len(edges_m):,}")

edges_scoped = edges_m.iloc[scoped_edge_positions].reset_index(drop=True)

TOL_MAX = 10.0
log(f"Running bulk dwithin spatial-index self-join on scoped edges at {TOL_MAX} m...")
left, right = edges_scoped.sindex.query(edges_scoped.geometry, predicate="dwithin", distance=TOL_MAX)
log(f"raw candidate pairs (incl self/dup/adjacent-shared-node): {len(left):,}")

mask = left < right
left, right = left[mask], right[mask]
log(f"after i<j dedupe: {len(left):,}")

osm_id_arr = edges_scoped["osm_id"].to_numpy()
u_arr = edges_scoped["u"].to_numpy()
v_arr = edges_scoped["v"].to_numpy()

diff_osm = osm_id_arr[left] != osm_id_arr[right]
left, right = left[diff_osm], right[diff_osm]
log(f"after different-osm_id filter: {len(left):,}")

shared_node = (
    (u_arr[left] == u_arr[right])
    | (u_arr[left] == v_arr[right])
    | (v_arr[left] == u_arr[right])
    | (v_arr[left] == v_arr[right])
)
left, right = left[~shared_node], right[~shared_node]
log(f"after no-shared-node filter (remaining candidate pairs to inspect exactly): {len(left):,}")

[11:50:37] Reprojected edges to metric CRS (EPSG:32617) for Phase 3 distance operations.


[11:50:41] edges within 300.0m of any fragile origin: 1,243,179 of 1,856,748


[11:50:42] Running bulk dwithin spatial-index self-join on scoped edges at 10.0 m...


[11:50:57] raw candidate pairs (incl self/dup/adjacent-shared-node): 7,195,995


[11:50:57] after i<j dedupe: 2,976,408


[11:50:57] after different-osm_id filter: 1,445,721


[11:50:57] after no-shared-node filter (remaining candidate pairs to inspect exactly): 694,053


In [13]:
log("Computing exact distances + endpoint classification for candidate pairs...")
geoms = edges_scoped.geometry.to_numpy()
ENDPOINT_EPS = 0.5  # meters; within this of a segment's own endpoint counts as "at the endpoint"

records = []
for i, j in zip(left, right):
    gi, gj = geoms[i], geoms[j]
    dist = gi.distance(gj)
    if dist > TOL_MAX:
        continue
    pi, pj = nearest_points(gi, gj)
    gi_coords = list(gi.coords)
    gj_coords = list(gj.coords)
    i_at_endpoint = any(pi.distance(Point(c)) <= ENDPOINT_EPS for c in gi_coords)
    j_at_endpoint = any(pj.distance(Point(c)) <= ENDPOINT_EPS for c in gj_coords)
    records.append(
        {
            "osm_id_i": osm_id_arr[i],
            "osm_id_j": osm_id_arr[j],
            "distance_m": float(dist),
            "i_at_endpoint": i_at_endpoint,
            "j_at_endpoint": j_at_endpoint,
            "non_endpoint_candidate": not (i_at_endpoint and j_at_endpoint),
            "mid_x": (pi.x + pj.x) / 2.0,
            "mid_y": (pi.y + pj.y) / 2.0,
        }
    )
candidates = pd.DataFrame.from_records(records)
log(f"candidates={len(candidates):,}")

candidates["within_2m"] = candidates["distance_m"] <= 2.0
candidates["within_10m"] = candidates["distance_m"] <= 10.0

print("Candidate counts by tolerance x endpoint classification:")
display(
    candidates.groupby(["within_2m", "within_10m", "non_endpoint_candidate"]).size().rename("n").reset_index()
)

# Diagnostic: are these dominated by a handful of osm_id pairs running
# parallel for a long stretch (e.g. divided-highway carriageways), or mostly
# distinct, one-off near-touches?
non_endpoint_10m = candidates.loc[candidates["non_endpoint_candidate"] & candidates["within_10m"]]
pair_counts = non_endpoint_10m.groupby(["osm_id_i", "osm_id_j"]).size()
big_pairs = pair_counts[pair_counts >= 20]
print(f"\nDistinct osm_id pairs among non-endpoint <=10m candidates: {len(pair_counts):,}")
print(
    f"Of those, {len(big_pairs)} pairs appear >=20 times (long parallel runs), "
    f"covering {int(big_pairs.sum()):,} of {len(non_endpoint_10m):,} candidate rows "
    f"({big_pairs.sum() / len(non_endpoint_10m) * 100:.1f}%) -- i.e. NOT dominated by long parallel-carriageway pairs."
)

[11:50:57] Computing exact distances + endpoint classification for candidate pairs...


[11:52:03] candidates=694,053


Candidate counts by tolerance x endpoint classification:


,within_2m,within_10m,non_endpoint_candidate,n
0,False,True,False,536594
1,False,True,True,142366
2,True,True,False,9995
3,True,True,True,5098



Distinct osm_id pairs among non-endpoint <=10m candidates: 73,361
Of those, 117 pairs appear >=20 times (long parallel runs), covering 3,421 of 147,464 candidate rows (2.3%) -- i.e. NOT dominated by long parallel-carriageway pairs.


In [14]:
# Proximity check: is each candidate near a baseline-fragile block's origin
# node, and if so, is that block dead_end or bridge_limited (Phase 1 label)?
# Simplification flagged: this checks proximity to the fragile block's origin
# node only, not a full shortest-path buffer toward its nearest service.
frag_coords = mech_df.merge(
    nodes[["node_id", "x", "y"]], left_on="origin_node_id", right_on="node_id", how="left"
)
frag_tree = cKDTree(frag_coords[["x", "y"]].to_numpy())
cand_xy = candidates[["mid_x", "mid_y"]].to_numpy()
dist_to_frag, idx_to_frag = frag_tree.query(cand_xy, k=1)
candidates["nearest_fragile_block_geoid"] = frag_coords["block_geoid"].to_numpy()[idx_to_frag]
candidates["nearest_fragile_mechanism"] = frag_coords["mechanism"].to_numpy()[idx_to_frag]
candidates["distance_to_nearest_fragile_m"] = dist_to_frag
candidates["near_fragile_block"] = candidates["distance_to_nearest_fragile_m"] <= 100.0

print("near_fragile_block (<=100m) x nearest_fragile_mechanism, non-endpoint candidates within 10m:")
subset_10 = candidates.loc[candidates["within_10m"] & candidates["non_endpoint_candidate"]]
display(pd.crosstab(subset_10["near_fragile_block"], subset_10["nearest_fragile_mechanism"], margins=True))

print("\nnear_fragile_block (<=100m) x nearest_fragile_mechanism, non-endpoint candidates within 2m:")
subset_2 = candidates.loc[candidates["within_2m"] & candidates["non_endpoint_candidate"]]
display(pd.crosstab(subset_2["near_fragile_block"], subset_2["nearest_fragile_mechanism"], margins=True))

candidates.to_parquet(OUTPUT_DIR / "phase3_unplanarized_crossing_candidates.parquet", index=False)
candidates.to_csv(OUTPUT_DIR / "phase3_unplanarized_crossing_candidates.csv", index=False)
log(f"Saved Phase 3 candidate table to {OUTPUT_DIR}")

near_fragile_block (<=100m) x nearest_fragile_mechanism, non-endpoint candidates within 10m:


nearest_fragile_mechanism,bridge_limited,dead_end,All
near_fragile_block,,,
False,52278,61876,114154
True,14324,18986,33310
All,66602,80862,147464



near_fragile_block (<=100m) x nearest_fragile_mechanism, non-endpoint candidates within 2m:


nearest_fragile_mechanism,bridge_limited,dead_end,All
near_fragile_block,,,
False,1938,2329,4267
True,411,420,831
All,2349,2749,5098


[11:52:08] Saved Phase 3 candidate table to C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism


### Phase 3 findings

**Raw candidate counts are large, but not dominated by the mechanism that
would matter (long parallel-carriageway pairs), and shrink sharply once
scoped to genuine fragile-block proximity.** At 10 m, 147,464 non-endpoint
candidate pairs; at the tighter, more-likely-genuine 2 m tolerance, 5,098.
These are **not** dominated by a handful of osm_id pairs running parallel for
a long stretch (e.g. divided-highway carriageways): 147,464 candidate rows
come from 73,361 *distinct* osm_id pairs, and pairs appearing >=20 times
(a long-parallel-run signature) cover only 2.3% of candidates. The more
likely dominant source is the ordinary OSM pattern of `highway=service`
driveways/parking-aisle access ways digitized to run a few meters alongside
a main road before their actual connecting node, not a systematic
planarization failure.

**Most candidates are nowhere near a fragile block.** Of the 147,464
non-endpoint candidates within 10 m, only 33,310 (22.6%) sit within 100 m of
a fragile block's origin node; at the tighter 2 m tolerance, only 831 of
5,098 (16.3%) do. The great majority of raw candidates are simply irrelevant
to the fragile classification.

**No bridge_limited concentration -- essentially even, same as Phase 2.**
Among the 831 near-fragile, 2 m-tolerance candidates: 411 (49.5%)
`bridge_limited`, 420 (50.5%) `dead_end`. Among the 33,310 near-fragile, 10 m
candidates: 14,324 (43.0%) `bridge_limited`, 18,986 (57.0%) `dead_end` --
if anything skewed slightly toward `dead_end`, not `bridge_limited`.

**Answering the deferral question plainly:** deferring full OSRM-based
network revalidation (`docs/manuscript_feedback_todo.md` item 4) looks
**safe for the baseline fragile headline number specifically**. Even at the
more permissive 2 m tolerance, only 831 candidate locations sit near a
fragile block at all -- 4.8% of the 17,335-block fragile universe -- and
they do not concentrate in `bridge_limited` in a way that would suggest
missed crossings are manufacturing spurious causeway-limited cases. That
said, 831 is not nothing, and a handful of genuine missed crossings inside
that set would be worth a targeted (not urgent, not full-network) follow-up
look if this notebook's mechanism finding is ever challenged in review --
the Phase 4 `_crossing_adjacent` sample below is exactly the starting point
for that.

## Phase 4 -- Stratified spot-check export

A small stratified sample (~20 per mechanism: `dead_end`, `bridge_limited`,
and a `_crossing_adjacent` subset of fragile blocks sitting within 100 m of a
non-endpoint Phase-3 candidate within 10 m) for later manual imagery
spot-checking. Not attempting the imagery check itself here -- just producing
the exportable sample, reprojected to lon/lat (`OGC:CRS84`) with
two-edge-component size included.

In [15]:
node_xy = nodes.set_index("node_id")[["x", "y"]]

sample_frames = []
for mech in ["dead_end", "bridge_limited"]:
    subset = mech_df.loc[mech_df["mechanism"] == mech].copy()
    n = min(20, len(subset))
    sample_frames.append(subset.sample(n=n, random_state=42))
    log(f"{mech}: sampled {n} of {len(subset):,}")

crossing_geoids = candidates.loc[
    candidates["within_10m"] & candidates["non_endpoint_candidate"] & candidates["near_fragile_block"],
    "nearest_fragile_block_geoid",
].unique()
crossing_adjacent = mech_df.loc[mech_df["block_geoid"].isin(crossing_geoids)].copy()
crossing_adjacent["mechanism"] = crossing_adjacent["mechanism"] + "_crossing_adjacent"
n_crossing = min(20, len(crossing_adjacent))
if n_crossing > 0:
    sample_frames.append(crossing_adjacent.sample(n=n_crossing, random_state=42))
log(f"crossing_adjacent: sampled {n_crossing} of {len(crossing_adjacent):,}")

sample = pd.concat(sample_frames, ignore_index=True)
sample = sample.join(node_xy, on="origin_node_id")

transformer = Transformer.from_crs("EPSG:32617", "OGC:CRS84", always_xy=True)
lon, lat = transformer.transform(sample["x"].to_numpy(), sample["y"].to_numpy())
sample["origin_lon"] = lon
sample["origin_lat"] = lat

export_cols = [
    "block_geoid", "county_name", "mechanism", "origin_node_id",
    "origin_lon", "origin_lat", "two_edge_component_size",
]
sample_export = sample[export_cols].copy()

csv_path = OUTPUT_DIR / "phase4_spotcheck_sample.csv"
sample_export.to_csv(csv_path, index=False)
log(f"Saved {csv_path} ({len(sample_export)} rows)")

sample_gdf = gpd.GeoDataFrame(
    sample_export,
    geometry=gpd.points_from_xy(sample_export["origin_lon"], sample_export["origin_lat"]),
    crs="OGC:CRS84",
)
geojson_path = OUTPUT_DIR / "phase4_spotcheck_sample.geojson"
sample_gdf.to_file(geojson_path, driver="GeoJSON")
log(f"Saved {geojson_path}")

display(sample_export["mechanism"].value_counts())
display(sample_export)

[11:52:08] dead_end: sampled 20 of 8,877


[11:52:08] bridge_limited: sampled 20 of 8,458


[11:52:09] crossing_adjacent: sampled 20 of 5,726


[11:52:09] Saved C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism\phase4_spotcheck_sample.csv (60 rows)


[11:52:09] Saved C:\Users\Vivek\Dropbox\repos\slr_fl_fragile_access\data\processed\analysis\baseline_fragility_mechanism\phase4_spotcheck_sample.geojson

mechanism
dead_end                            20
bridge_limited                      20
bridge_limited_crossing_adjacent    11
dead_end_crossing_adjacent           9
Name: count, dtype: int64

,block_geoid,county_name,mechanism,origin_node_id,origin_lon,origin_lat,two_edge_component_size
0,120990076222010,Palm Beach,dead_end,351778,-80.132217,26.357263,1
1,120860106132002,Miami-Dade,dead_end,1649135,-80.319546,25.571441,1
2,120860017041007,Miami-Dade,dead_end,759942,-80.243357,25.810259,1
3,120860010033007,Miami-Dade,dead_end,891748,-80.234202,25.853162,1
4,120860039121012,Miami-Dade,dead_end,238904,-80.128973,25.866575,1
5,120860059011001,Miami-Dade,dead_end,889821,-80.290974,25.763290,1
6,120860042081002,Miami-Dade,dead_end,1631821,-80.143502,25.791106,1
7,120860106182006,Miami-Dade,dead_end,1651424,-80.334649,25.563020,1
8,120990059263001,Palm Beach,dead_end,1477044,-80.129751,26.512096,1
9,120990075041014,Palm Beach,dead_end,1366729,-80.078849,26.327843,1


## Summary

**Phase 1 (structural mechanism, production-faithful graph).** The 17,335
baseline-fragile blocks split almost exactly evenly: **51.21% `dead_end`
(degree <= 1) and 48.79% `bridge_limited`** (degree >= 2 but the node's
two-edge-component has no service), with zero blocks unexplained by either
mechanism. This is not geographically uniform -- Miami-Dade skews `dead_end`
(61.5%), Palm Beach skews `bridge_limited` (57.4%), Broward is close to
50/50 -- consistent with each county's road geometry (dense interior grid vs.
barrier-island causeways). `bridge_limited` blocks are concentrated at small
two-edge-component sizes (57% in components of 10 nodes or fewer -- genuine
thin causeways/peninsulas) but with a real tail into 11-1000+-node
neighborhood-scale components, both legitimate instances of the same
mechanism.

**Phase 2 (private-road toggle).** Retaining `access=private` roads does not
shrink the fragile share -- it *grows* it slightly (24.85% to 27.48% on the
simplified comparison graph), because it reclassifies blocks in both
directions (rescuing some `isolated` blocks into `fragile`, promoting some
`fragile` blocks to `redundant`, and vice versa for a smaller number).
**The task's working hypothesis -- that retaining private roads would
disproportionately rescue `bridge_limited` blocks rather than `dead_end`
ones -- is not supported by the data.** Among blocks that lose `fragile`
status when private roads are retained, it is close to a coin flip: 50.08%
`bridge_limited`, 49.92% `dead_end`.

**Phase 3 (unplanarized-crossing audit, candidate-detection only).** Raw
candidate counts are large (147,464 non-endpoint pairs at 10 m) but not
explained by long parallel-carriageway runs (only 2.3% of candidates come
from osm_id pairs repeating >=20 times); most likely reflect ordinary
`highway=service` driveways digitized near a main road. Only 16-23% of
candidates (depending on tolerance) sit within 100 m of any fragile block at
all, and among those, again no `bridge_limited` concentration (49.5%/50.5%
at 2 m). Deferring full OSRM-based network revalidation looks **safe for the
baseline fragile headline number specifically**, though not something to
declare closed forever -- the absolute count (831 near-fragile candidates at
2 m) is small relative to 17,335 fragile blocks but not zero.

**Phase 4.** A stratified ~20-per-mechanism sample (`dead_end`,
`bridge_limited`, and a `_crossing_adjacent` subset from Phase 3) was
exported to `data/processed/analysis/baseline_fragility_mechanism/` as CSV
and GeoJSON (lon/lat, `OGC:CRS84`) for manual imagery spot-checking --  not
attempted here.

**What is confirmed:** the flat baseline fragile share is a real,
well-explained structural feature of the road network, not a bug or a
single-cause artifact. It decomposes cleanly and almost evenly into two
distinct, well-understood mechanisms (dead ends and causeway/bridge-limited
access), the split is geographically patterned in a way consistent with each
county's actual road geometry, and neither the private-road filter nor
plausible unplanarized-crossing gaps materially favor one mechanism over the
other or threaten the headline count.

**What remains ambiguous:** whether specific individual `bridge_limited`
blocks in the component-size tail (11-1000+ nodes) are genuinely
service-desert neighborhoods or partly an artifact of where services happen
to be snapped; and whether any of the 831 tight-tolerance,
near-fragile Phase 3 candidates are real missed crossings (this notebook
only flags candidates -- it does not resolve them).

**Deviations from the task brief, flagged plainly:**
- Phase 1/3 deliberately used the *production-faithful* graph construction
  (bridge_rule="approach", origin nodes from the verified parquet,
  raw-graph-2ecc-eligible service snapping) rather than the brief's literal
  "setup sequence to mirror" (which is `02d`'s simplified, bridge-rule-free
  construction) -- justified in Phase 0b, and validated live against
  `run_manifest.json`'s exact bridge-removal counts.
- Phase 3's first implementation had a real, fixed bug: `edges.geometry`
  from `segmentize_roads()` is in `OGC:CRS84` (degrees), not the projected
  metric CRS, so an initial unscoped `dwithin(..., distance=10.0)` was
  silently 10 *degrees* and drove free system RAM from 32 GB down to
  ~320 MB before being killed and fixed (reproject to the metric CRS, then
  scope to a 300 m buffer around fragile-block origins before the
  self-join).
- `fragility_sensitivity_comparison.csv` does not currently exist on this
  machine (the `--sensitivity` run that would produce it was never run), so
  Phase 2's `reported_baseline` row was built directly from the verified
  parquet rather than read from that file, per the task brief's fallback
  instruction.
- No number failed to reproduce; the 68,521/17,335/25.30% ground truth was
  confirmed exactly, and the production bridge-removal counts (3,482
  structures, 200 removed at 0 ft) were reproduced exactly as well.

**Draft sentence for whoever writes up the methods section** (draft only,
not manuscript-final): *"Baseline (0 ft) fragile access is not driven by a
single mechanism: it splits almost evenly between literal dead-end street
segments (51%) and single-causeway/bridge-limited access to an otherwise
well-connected network (49%), with the balance shifting by county in a way
consistent with local road geometry (dead ends more common in Miami-Dade's
dense interior grid, causeway-limited access more common along Palm Beach's
barrier-island geography); this split is stable under an
OSM-private-road-inclusion sensitivity check and does not appear to be an
artifact of unplanarized road-network crossings."*